# Interview Question Generator — GPT API Mini Project

**Use case (Project idea #15):** The user gives a **job role**, and the app returns
**10 tailored interview questions with sample answers**.

**Skill demonstrated:** domain-specific generation + structured output.
  
**API used:** Groq (free, OpenAI-compatible Chat Completions API)

---
This notebook walks through:
1. Setting up the API key and client
2. A first Chat Completions call
3. Building the core question-generation function
4. Getting clean **structured (JSON)** output we can reuse in an app
5. Experimenting with **parameters** (`temperature`, `max_tokens`, `top_p`) and recording findings
6. Best practices we applied

A companion **Streamlit app** (`interview_question_app.py`) wraps the same logic in a simple UI.

## 1. Setup: API key & client

We use **Groq** — a free, no-credit-card LLM API that is OpenAI-compatible,
so we use the same `openai` Python package and just point it at Groq's endpoint.

Get a free key at **console.groq.com** (Google/GitHub login, no card needed).

Keep the key out of the code in a `.env` file (never commit it). It should contain:

```
GROQ_API_KEY=gsk_your-key-here
```

`load_dotenv()` reads that file into the environment.

In [5]:
# Groq is OpenAI-compatible, so we use the same openai package
%pip install --upgrade openai python-dotenv -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

print("API key loaded:", bool(os.getenv("GROQ_API_KEY")))

API key loaded: True


We will use one model throughout. `gpt-4o-mini` is a good cheap default;
you can swap it for `gpt-3.5-turbo` or `gpt-4o` by changing this one variable.

In [7]:
MODEL = "llama-3.3-70b-versatile"   

## 2. A first Chat Completions call

The Chat Completions API takes a **list of messages**, each with a `role`
(`system`, `user`, or `assistant`) and `content`. The `system` message sets the
assistant's behaviour; the `user` message is the request.

In [8]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an experienced technical recruiter."},
        {"role": "user", "content": "Give me one good interview question for a Data Analyst."},
    ],
)

print(response.choices[0].message.content)

Here's a good interview question for a Data Analyst:

**Question:** "Suppose you're analyzing sales data for an e-commerce company, and you notice that sales have been declining over the past quarter. However, when you drill down into the data, you see that sales for one particular product category have actually been increasing. What steps would you take to investigate this discrepancy, and what insights might you hope to gain from your analysis?"

**Why it's a good question:**

* This question tests the candidate's ability to think critically and analytically about complex data.
* It assesses their ability to identify interesting patterns or anomalies in the data and develop hypotheses to explain them.
* The question also gives the candidate an opportunity to walk you through their analytical process, including the types of questions they would ask, the data they would gather, and the insights they would aim to derive.
* Finally, the question allows you to evaluate the candidate's com

## 3. The core function: generate interview questions
The **system prompt** is where we encode all
the behaviour we want — this is the "customized prompt" that turns a generic
chatbot into a focused interview-question tool.

Key design choices in the prompt:
- It must always produce **exactly 10** questions.
- Each question pairs with a **concise sample answer**.
- It should **mix question types** (technical, behavioural, situational) so the
  set feels like a real interview.
- It should adapt difficulty to a **seniority level** the user can choose.

In [9]:
SYSTEM_PROMPT = """You are an experienced hiring manager and technical recruiter.
Your job is to create realistic interview questions for a given job role.

When given a JOB ROLE (and optionally a seniority level), produce exactly 10
interview questions tailored to that role. For each question, also write a
concise model answer (3-5 sentences) that a strong candidate might give.

Requirements:
- Mix the question types: include technical/role-specific, behavioural
  ("Tell me about a time..."), and situational ("What would you do if...") questions.
- Order questions roughly from warm-up to more challenging.
- Match difficulty to the requested seniority (default to mid-level if none given).
- Keep questions clear and free of bias; do not ask about age, religion,
  marital status, or other protected characteristics.
- Keep sample answers practical and specific to the role, not generic filler."""


def generate_interview_questions(role, seniority="mid-level", **params):
    """Generate interview questions + sample answers for a role.

    Extra keyword args (temperature, max_tokens, top_p, ...) are passed
    straight through to the API so we can experiment with them.
    """
    user_message = f"Job role: {role}\nSeniority level: {seniority}"

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        **params,
    )
    return completion.choices[0].message.content

In [10]:
# Try it out
output = generate_interview_questions("Junior Data Analyst", seniority="entry-level")
print(output)

Here are 10 interview questions tailored to the Junior Data Analyst role at an entry-level:

1. **What inspired you to pursue a career in data analysis, and what do you hope to achieve in this field?**
A strong candidate might say: "I've always been fascinated by the power of data to tell stories and inform decisions. As a Junior Data Analyst, I hope to develop my skills in working with datasets, creating visualizations, and communicating insights to stakeholders. I'm excited to learn and grow in this role, and I believe my strong work ethic and attention to detail will serve me well. I'm looking forward to working with a team to drive business outcomes through data-driven decision-making."

2. **Can you walk me through your experience with data visualization tools, such as Tableau or Power BI?**
A strong candidate might say: "In my coursework, I had the opportunity to work with Tableau, creating interactive dashboards and visualizations for a mock business scenario. I enjoyed learning

## 4. Structured output (JSON)

Free-form text is nice to read, but if we want to use the questions in an app
(loop over them, show one at a time, store them), **structured output** is far
easier to work with. We ask the model to return JSON and use
`response_format={"type": "json_object"}` so the reply is guaranteed to be valid JSON.

In [11]:
JSON_SYSTEM_PROMPT = SYSTEM_PROMPT + """

Return ONLY a valid JSON object with this exact shape:
{
  "role": "<the job role>",
  "seniority": "<the seniority level>",
  "questions": [
    {"type": "technical|behavioural|situational",
     "question": "<the question>",
     "sample_answer": "<a concise model answer>"}
  ]
}
The "questions" array must contain exactly 10 items. Do not add any text outside the JSON."""


def generate_questions_json(role, seniority="mid-level", **params):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JSON_SYSTEM_PROMPT},
            {"role": "user", "content": f"Job role: {role}\nSeniority level: {seniority}"},
        ],
        response_format={"type": "json_object"},
        **params,
    )
    return json.loads(completion.choices[0].message.content)

In [12]:
data = generate_questions_json("Data Analyst", seniority="mid-level")

print("Role:", data["role"], "| Seniority:", data["seniority"])
print("Number of questions:", len(data["questions"]))
print()

# Show the first 3 in a clean format
for i, q in enumerate(data["questions"][:3], start=1):
    print(f"Q{i} [{q['type']}]: {q['question']}")
    print(f"   Sample answer: {q['sample_answer']}\n")

Role: Data Analyst | Seniority: mid-level
Number of questions: 10

Q1 [technical]: What is the difference between a histogram and a bar chart, and when would you use each?
   Sample answer: A histogram is a type of bar chart used to represent continuous data, while a bar chart is used for categorical data. I would use a histogram to show the distribution of numerical data, such as exam scores, and a bar chart to compare categorical data, such as sales by region. Understanding the difference is crucial for effective data visualization. In my previous role, I used histograms to analyze customer demographics and bar charts to compare sales performance across different product categories.

Q2 [behavioural]: Tell me about a time when you had to analyze a complex dataset and identify key trends or insights. How did you approach the task?
   Sample answer: In my previous role, I was tasked with analyzing customer purchase behavior. I started by cleaning and preprocessing the data, then used s

## 5. Experimenting with parameters

The project asks us to vary parameters and observe the effect. The two most
important for this use case are **`temperature`** and **`max_tokens`**.

- **`temperature`** (0–2): lower = more focused/deterministic, higher = more
  diverse/creative (and eventually incoherent).
- **`max_tokens`**: caps the response length. Too low and the answer gets cut off
  (you'll see `finish_reason == "length"`).


### Companion app
`interview_question_app.py` is a Streamlit front-end that reuses this exact prompt.
Run it with:

```bash
streamlit run interview_question_app.py
```